# 🎬 Antigravity ➔ Google Colab 遠端 GPU 影片生成中樞

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MazingerSam/video_test/blob/main/notebooks/colab_ai_video_workflow.ipynb)

本筆記本提供 **Antigravity 專屬遠端 GPU 運算伺服器**。
- 🧠 **GPU 算力引擎**：直接利用 Colab 免費 T4 GPU 運行 **AnimateDiff** (文字生影) 與 **Stable Video Diffusion** (圖片生影)。
- 📡 **即時橋接**：透過內建的 Cloudflare Tunnel 穿透技術，產生公開專屬 API 連線網址。
- 🤖 **Agent 全自動化**：您只要在本地 Antigravity 提出提示詞，Antigravity 會自動調用本筆記本的 GPU 算力，並將 MP4 檔案自動傳回本地電腦！

## 步驟 0：檢查顯卡環境
> **重要提示**：在上方選單選擇 **「執行階段 (Runtime)」➔「變更執行階段類型」➔ 確認選擇「T4 GPU」**。

In [ ]:
!nvidia-smi

## 步驟 1：安裝核心套件與伺服器模組

In [ ]:
# 安裝 AI 推論與橋接伺服器套件
!pip install -q -U diffusers transformers accelerate imageio[ffmpeg] safetensors fastapi uvicorn pycloudflared pydantic nest_asyncio

## 步驟 2：載入開源模型至 GPU 顯存 (常駐顯存，加速後續生成)

In [ ]:
import torch
from diffusers import AnimateDiffPipeline, MotionAdapter, DDIMScheduler, StableVideoDiffusionPipeline

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print("⏳ [1/2] 正在載入 AnimateDiff 模型 (文字轉影片)... ")
adapter = MotionAdapter.from_pretrained("guoyww/animatediff-motion-adapter-v1-5-2", torch_dtype=dtype)
pipe_t2v = AnimateDiffPipeline.from_pretrained(
    "SG161222/Realistic_Vision_V5.1_noVAE",
    motion_adapter=adapter,
    torch_dtype=dtype
)
pipe_t2v.scheduler = DDIMScheduler.from_config(
    pipe_t2v.scheduler.config,
    beta_schedule="linear",
    clip_sample=False,
    timestep_spacing="linspace",
    steps_offset=1
)
pipe_t2v.enable_vae_slicing()
pipe_t2v.enable_model_cpu_offload()

print("⏳ [2/2] 正在載入 Stable Video Diffusion XT (圖片轉影片)... ")
pipe_i2v = StableVideoDiffusionPipeline.from_pretrained(
    "stabilityai/stable-video-diffusion-img2vid-xt",
    torch_dtype=dtype,
    variant="fp16" if torch.cuda.is_available() else None
)
pipe_i2v.enable_vae_slicing()
pipe_i2v.enable_model_cpu_offload()

print("✨ 兩大開源 AI 影片模型已成功常駐 GPU 顯存！")

## 步驟 3：啟動 Antigravity 遠端 GPU 橋接伺服器
執行以下儲存格後，會產生一組公開的 `https://xxxx.trycloudflare.com` 連線網址。
請將該網址貼給 Antigravity 或執行本地命令，即可開始遠端遙控！

In [ ]:
import os
import time
import base64
import io
import threading
from pathlib import Path
from PIL import Image
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from diffusers.utils import export_to_video
from pycloudflared import try_cloudflare
import uvicorn

app = FastAPI(title="Antigravity Colab GPU Bridge")
OUTPUT_DIR = Path("./server_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class Text2VideoRequest(BaseModel):
    prompt: str
    negative_prompt: str = "bad quality, blurry, distorted, lowres"
    num_frames: int = 16
    num_inference_steps: int = 25
    guidance_scale: float = 7.5

class Image2VideoRequest(BaseModel):
    image_base64: str
    num_frames: int = 25
    fps: int = 7
    motion_bucket_id: int = 127

@app.get("/health")
def health():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {
        "status": "online",
        "gpu": gpu_name,
        "models_loaded": ["AnimateDiff", "StableVideoDiffusion-XT"]
    }

@app.post("/generate/text2video")
def generate_text2video(req: Text2VideoRequest):
    try:
        print(f"[*] 收到文字生影請求: {req.prompt}")
        output = pipe_t2v(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            num_frames=req.num_frames,
            guidance_scale=req.guidance_scale,
            num_inference_steps=req.num_inference_steps,
            generator=torch.Generator(device=device).manual_seed(int(time.time()))
        )
        temp_file = OUTPUT_DIR / f"t2v_{int(time.time())}.mp4"
        export_to_video(output.frames[0], str(temp_file), fps=8)
        
        with open(temp_file, "rb") as f:
            b64_data = base64.b64encode(f.read()).decode("utf-8")
        return {"status": "success", "video_base64": b64_data, "filename": temp_file.name}
    except Exception as e:
        print(f"[!] 生成出錯: {e}")
        return {"status": "error", "error": str(e)}

@app.post("/generate/image2video")
def generate_image2video(req: Image2VideoRequest):
    try:
        print("[*] 收到圖片生影請求 (SVD-XT)")
        img_bytes = base64.b64decode(req.image_base64)
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB").resize((1024, 576))
        
        generator = torch.manual_seed(int(time.time()))
        frames = pipe_i2v(
            img,
            decode_chunk_size=8,
            generator=generator,
            motion_bucket_id=req.motion_bucket_id,
            num_frames=req.num_frames
        ).frames[0]
        
        temp_file = OUTPUT_DIR / f"i2v_{int(time.time())}.mp4"
        export_to_video(frames, str(temp_file), fps=req.fps)
        
        with open(temp_file, "rb") as f:
            b64_data = base64.b64encode(f.read()).decode("utf-8")
        return {"status": "success", "video_base64": b64_data, "filename": temp_file.name}
    except Exception as e:
        print(f"[!] 生成出錯: {e}")
        return {"status": "error", "error": str(e)}

# 背景啟動 Uvicorn
def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

threading.Thread(target=start_server, daemon=True).start()
time.sleep(2)

# 透過 Cloudflare Tunnel 建立外部公開安全連線
tunnel = try_cloudflare(port=8000)
colab_public_url = tunnel.tunnel_url

print("\n" + "="*60)
print("🎉 Antigravity 遠端 GPU 伺服器已成功上線！")
print(f"🔗 專屬連線網址: {colab_public_url}")
print("="*60)
print("\n👉 請將上方連線網址貼給 Antigravity，或者在本機執行：")
print(f"   python src/colab_client.py connect {colab_public_url}\n")